# 06 — Demo: Inference Pipeline
CS2 Match Outcome Predictor & Seeding Engine

Given N team names + tournament type, this notebook:
1. Fuzzy-matches team names to the dataset
2. Builds a "current state" feature snapshot for each team
3. Predicts pairwise win probabilities (with symmetry correction)
4. Aggregates into a power score and seed order
5. Generates a bracket

In [1]:
import os

# === For Colab: clone the repo if not already present ===
if not os.path.exists('CS2-Capstone-Project') and 'CS2-Capstone-Project' not in os.getcwd():
    try:
        import google.colab
        !pip install rapidfuzz -q
        !git clone https://github.com/AzizkhonM/CS2-Capstone-Project.git
        os.chdir('CS2-Capstone-Project')
        print("Colab: repo clone qilindi va CWD o'zgartirildi")
    except ImportError:
        pass

# === For VS Code / local Jupyter: find repo root automatically ===
def find_repo_root(marker='requirements.txt'):
    path = os.getcwd()
    while True:
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root (looking for '{marker}')")
        path = parent

repo_root = find_repo_root()
os.chdir(repo_root)

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

import sys, pickle
sys.path.append('.')

import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

import logging, time
logging.Formatter.converter = lambda *args: time.localtime(time.time() + 5*3600)
logging.basicConfig(
    filename='data/06_demo.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    filemode='w',
    force=True
)

print("Working directory set to:", os.getcwd())

Cloning into 'CS2-Capstone-Project'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 54 (delta 19), reused 48 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 204.70 KiB | 4.01 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/CS2-Capstone-Project


In [2]:
with open('models/logreg_model.pkl', 'rb') as f:
    logreg = pickle.load(f)
with open('models/feature_list.pkl', 'rb') as f:
    final_features = pickle.load(f)

df = pd.read_csv('data/matches_with_elo.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

elo_ratings = pd.read_csv('models/elo_ratings.csv').set_index('team_name')['elo_rating'].to_dict()

print(f"Loaded model, {len(final_features)} features, {len(df)} matches, {len(elo_ratings)} team Elo ratings")
logging.info(f"Loaded model/data: {len(elo_ratings)} teams")

Loaded model, 20 features, 6989 matches, 331 team Elo ratings


## 1. Rebuild reframed columns (needed to extract per-team snapshots)

In [3]:
from src.features import build_features, REFRAME_BASES, COUNT_BASES

df, final_features_check = build_features(df)

assert set(final_features) == set(final_features_check), "Feature mismatch!"

print("Features rebuilt. Diff columns ready.")
print(df[final_features].head(2))

Features rebuilt. Diff columns ready.
   elo_diff  past3_diff  head2head_percentage_diff  head2head_freq_diff  \
0       0.0       -8.57                        0.0                  0.0   
1       0.0        8.48                       60.0                  3.0   

   mirage_diff  inferno_diff  nuke_diff  dust2_diff  overpass_diff  \
0         -1.8          -0.4       -2.8        19.9            2.3   
1         17.4          21.4       -3.6         5.2            0.7   

   train_diff  ancient_diff  vertigo_diff  anubis_diff  wins_diff  \
0         3.0          16.3          10.1         15.2         15   
1        -9.2          42.2          19.6         -2.2         15   

   losses_diff  totalwinrate_diff  online_winrate_diff  lan_winrate_diff  \
0          -14           0.405556            -0.442308          0.081007   
1           -1           0.154389             0.012656          0.192547   

   overall_winrate_diff  event_type_lan  
0                   0.0               1  
1   

## 2. Build a "current state" snapshot per team

For each team, take its most recent match and extract its own
(non-diff) values for every dynamic feature. This represents "what we
know about this team today."

In [4]:
snapshot_cols = REFRAME_BASES + COUNT_BASES

def get_team_snapshot(team_name, df):
    t1_rows = df[df['team1_name'] == team_name]
    t2_rows = df[df['team2_name'] == team_name]

    combined = []
    for _, r in t1_rows.iterrows():
        combined.append((r['date'], {c: r[f'team1_{c}'] for c in snapshot_cols}))
    for _, r in t2_rows.iterrows():
        combined.append((r['date'], {c: r[f'team2_{c}'] for c in snapshot_cols}))

    if not combined:
        return None

    combined.sort(key=lambda x: x[0])
    return combined[-1][1]  # most recent snapshot

all_teams = sorted(pd.concat([df['team1_name'], df['team2_name']]).unique())
team_snapshots = {t: get_team_snapshot(t, df) for t in all_teams}

print(f"Built snapshots for {len(team_snapshots)} teams")
logging.info(f"Built snapshots for {len(team_snapshots)} teams")

Built snapshots for 331 teams


## 3. Fuzzy team-name matching

In [5]:
def resolve_team_name(input_name, known_teams, threshold=80):
    match, score, _ = process.extractOne(input_name, known_teams, scorer=fuzz.WRatio)
    if score >= threshold:
        return match, score
    return None, score

def resolve_teams(team_names, known_teams):
    resolved = {}
    for name in team_names:
        match, score = resolve_team_name(name, known_teams)
        resolved[name] = match
        if match:
            print(f"'{name}' -> '{match}' (score={score})")
        else:
            print(f"'{name}' -> NOT FOUND (best score={score}) — will use default/new-team fallback")
        logging.info(f"Resolved '{name}' -> '{match}' (score={score})")
    return resolved

## 4. Pairwise prediction (with symmetry correction)

In [6]:
from sklearn.preprocessing import StandardScaler

# Refit scaler on full training feature space (same as 04/05) for consistent scaling
train_X = df[final_features].fillna(0)
scaler = StandardScaler()
scaler.fit(train_X)

DEFAULT_ELO = 1500

def build_pair_features(team_a, team_b, event_type='lan'):
    snap_a = team_snapshots.get(team_a)
    snap_b = team_snapshots.get(team_b)

    row = {}
    elo_a = elo_ratings.get(team_a, DEFAULT_ELO)
    elo_b = elo_ratings.get(team_b, DEFAULT_ELO)
    row['elo_diff'] = elo_a - elo_b

    for base in REFRAME_BASES + COUNT_BASES:
        if base == 'totallossrate':
            continue
        if snap_a is None and snap_b is None:
            val_a, val_b = 0, 0
        elif snap_a is None:
            val_a = snap_b[base]
            val_b = snap_b[base]
        elif snap_b is None:
            val_a = snap_a[base]
            val_b = snap_a[base]
        else:
            val_a = snap_a[base]
            val_b = snap_b[base]
        row[f'{base}_diff'] = val_a - val_b

    row['event_type_lan'] = 1 if event_type == 'lan' else 0
    return pd.DataFrame([row])[final_features]

def predict_win_prob(team_a, team_b, event_type='lan'):
    """Returns symmetry-corrected P(team_a beats team_b)."""
    X_ab = build_pair_features(team_a, team_b, event_type)
    X_ab_scaled = pd.DataFrame(scaler.transform(X_ab), columns=final_features)
    p_ab = logreg.predict_proba(X_ab_scaled)[0][1]

    X_ba = build_pair_features(team_b, team_a, event_type)
    X_ba_scaled = pd.DataFrame(scaler.transform(X_ba), columns=final_features)
    p_ba = logreg.predict_proba(X_ba_scaled)[0][1]

    # Symmetry correction (see 05_evaluation.ipynb findings)
    p_final = (p_ab + (1 - p_ba)) / 2
    return p_final

## 5. Power score aggregation (Bradley-Terry style) + seeding

In [7]:
from itertools import combinations

def compute_power_scores(team_names, resolved_names, event_type='lan'):
    scores = {t: [] for t in team_names}
    pairwise = {}

    for a, b in combinations(team_names, 2):
        ra = resolved_names[a] or a
        rb = resolved_names[b] or b
        p_ab = predict_win_prob(ra, rb, event_type)
        pairwise[(a, b)] = p_ab
        scores[a].append(p_ab)
        scores[b].append(1 - p_ab)

    power_scores = {t: np.mean(v) for t, v in scores.items()}
    return power_scores, pairwise

def generate_seeding(team_names, event_type='lan'):
    resolved_names = resolve_teams(team_names, all_teams)
    power_scores, pairwise = compute_power_scores(team_names, resolved_names, event_type)
    ranked = sorted(power_scores.items(), key=lambda x: x[1], reverse=True)
    return ranked, pairwise

## 6. Bracket construction (standard sports rule, with byes)

In [8]:
def build_bracket(ranked_teams):
    n = len(ranked_teams)
    next_pow2 = 1
    while next_pow2 < n:
        next_pow2 *= 2

    seeded = [t for t, _ in ranked_teams] + [None] * (next_pow2 - n)  # None = bye

    pairs = []
    for i in range(next_pow2 // 2):
        top = seeded[i]
        bottom = seeded[next_pow2 - 1 - i]
        pairs.append((top, bottom))
    return pairs

## 7. Full demo run

In [9]:
teams = ["Spirit", "MyRandomTestTeam123", "G2", "Vitality", "FaZe", "MOUZ", "Liquid", "Astralis"]
tournament_type = "lan"

ranked, pairwise = generate_seeding(teams, tournament_type)

print("\n=== SEEDING RESULT ===")
for i, (team, score) in enumerate(ranked, 1):
    print(f"Seed {i}: {team:12s} (power score {score:.3f})")

bracket = build_bracket(ranked)
print("\n=== BRACKET ===")
for a, b in bracket:
    b_str = b if b else "BYE"
    print(f"  {a} vs {b_str}")

logging.info(f"Demo run: {len(teams)} teams, seeding: {[t for t,_ in ranked]}")

'Spirit' -> 'Spirit' (score=100.0)
'MyRandomTestTeam123' -> NOT FOUND (best score=60.00000000000001) — will use default/new-team fallback
'G2' -> 'G2' (score=100.0)
'Vitality' -> 'Vitality' (score=100.0)
'FaZe' -> 'FaZe' (score=100.0)
'MOUZ' -> 'MOUZ' (score=100.0)
'Liquid' -> 'Liquid' (score=100.0)
'Astralis' -> 'Astralis' (score=100.0)

=== SEEDING RESULT ===
Seed 1: Vitality     (power score 0.738)
Seed 2: Spirit       (power score 0.696)
Seed 3: MOUZ         (power score 0.528)
Seed 4: Liquid       (power score 0.480)
Seed 5: G2           (power score 0.465)
Seed 6: MyRandomTestTeam123 (power score 0.393)
Seed 7: FaZe         (power score 0.390)
Seed 8: Astralis     (power score 0.310)

=== BRACKET ===
  Vitality vs Astralis
  Spirit vs FaZe
  MOUZ vs MyRandomTestTeam123
  Liquid vs G2


## 8. Explanation for a single seed (feature-based)

In [10]:
def explain_seed(team, pairwise, teams):
    print(f"\nWhy {team} received its seed:")
    for (a, b), p_ab in pairwise.items():
        if team == a:
            print(f"  vs {b}: {p_ab:.1%} win probability")
        elif team == b:
            print(f"  vs {a}: {1-p_ab:.1%} win probability")

explain_seed("Vitality", pairwise, teams)


Why Vitality received its seed:
  vs Spirit: 54.8% win probability
  vs MyRandomTestTeam123: 67.6% win probability
  vs G2: 77.0% win probability
  vs FaZe: 82.3% win probability
  vs MOUZ: 72.2% win probability
  vs Liquid: 75.6% win probability
  vs Astralis: 87.1% win probability


In [11]:
try:
    from google.colab import files
    files.download('data/06_demo.log')
except ImportError:
    print("Not running in Colab.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>